# Fine-Grained Tool Calling

Combine **tool use + streaming** and you get tool arguments in real time. Alongside the familiar text deltas, you handle a new event type — **`input_json`** — with two properties:

- **`partial_json`** — the newest chunk of the tool-argument JSON
- **`snapshot`** — the cumulative JSON assembled so far

**The default: buffered + validated.** The API doesn't forward every keystroke. It waits for a *complete top-level key/value pair*, validates it against your schema, then releases the buffered chunks in a burst. That's why you see pause → burst → pause even with streaming on.

**Fine-grained tool calling** turns that buffering **off** by disabling server-side JSON validation:
- chunks arrive as fast as Claude generates them (no per-key delay)
- **you** must now cope with possibly-invalid JSON (e.g. `"word_count": undefined`)

Enable it with the beta flag `fine-grained-tool-streaming-2025-05-14`.

> **Adaptation:** the course uses `claude-sonnet-4-5`, but `chat_stream` always passes `temperature=1.0`, which **400s on flagship models** — so this notebook uses **`claude-haiku-4-5-20251001`** (supports temperature; consistent with the section). Streaming, tools, and the beta all work on Haiku. Note the classic "undefined" invalid-JSON output is **model-dependent** — a capable model may just emit valid JSON; the point is the *mechanism* and that you must handle invalid JSON when validation is off.

## Setup — streaming harness

`chat_stream` uses `client.beta.messages.stream` (so we can pass `betas`). The helpers are rebuilt to (de)serialize blocks as plain dicts, which is what streaming's assembled final message needs when we push it back into history.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
from anthropic import Anthropic
from anthropic.types import ToolParam

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def add_user_message(messages, message):
    if isinstance(message, list):
        messages.append({"role": "user", "content": message})
    else:
        messages.append({"role": "user", "content": [{"type": "text", "text": message}]})


def add_assistant_message(messages, message):
    if isinstance(message, list):
        messages.append({"role": "assistant", "content": message})
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append({
                    "type": "tool_use",
                    "id": block.id,
                    "name": block.name,
                    "input": block.input,
                })
        messages.append({"role": "assistant", "content": content_list})
    else:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": message}]})


def chat_stream(messages, system=None, temperature=1.0, stop_sequences=[],
                tools=None, tool_choice=None, betas=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tool_choice:
        params["tool_choice"] = tool_choice
    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system
    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Tool — `save_article`

A nested schema (a top-level `abstract` plus a `meta` object with `word_count` + `review`) so we can *see* the per-top-level-key buffering: `abstract` releases as one burst, then `meta` as another.

In [2]:
save_article_schema = ToolParam({
    "name": "save_article",
    "description": "Saves a scholarly journal article",
    "input_schema": {
        "type": "object",
        "properties": {
            "abstract": {"type": "string", "description": "Abstract of the article. One short sentence max"},
            "meta": {
                "type": "object",
                "properties": {
                    "word_count": {"type": "integer", "description": "Word count"},
                    "review": {"type": "string", "description": "Eight sentence review of the paper"},
                },
                "required": ["word_count", "review"],
            },
        },
        "required": ["abstract", "meta"],
    },
})


def save_article(**kwargs):
    return "Article saved!"


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []
    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            })
        except Exception as e:
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            })
    return tool_result_blocks

## Streaming conversation loop

We stream every chunk: text deltas, a header when a `tool_use` block starts, and each `input_json.partial_json` as it arrives. `fine_grained=True` adds the beta flag that disables validation. (`tool_choice` forces a specific tool and we break after one round so forcing doesn't loop forever.)

In [3]:
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

## Example 1 — default (validated) streaming

Watch the tool-argument JSON arrive in **bursts**: the whole `abstract` value appears at once, then the whole `meta` object. Those pauses are the API validating each top-level key/value pair before releasing its buffered chunks.

In [4]:
messages = []
add_user_message(messages, "Create and save a fake computer science article")

run_conversation(messages, tools=[save_article_schema])


>>> Tool Call: "save_article"
{"abstract": "A novel quantum-inspired algorithm for optimizing distributed neural networks through adaptive layer pruning.", "meta": {"word_count":8500,"review":"This paper presents an innovative approach to neural network optimization by combining quantum computing principles with classical distributed learning methods. The authors demonstrate their algorithm on several benchmark datasets, achieving notable improvements in both convergence speed and memory efficiency compared to existing baselines. The theoretical framework is well-grounded in both quantum mechanics and machine learning theory, providing solid mathematical foundations for the proposed method. Experimental results show consistent improvements across different network architectures and data distributions, with up to 40% reduction in training time on large-scale datasets. The work includes comprehensive ablation studies that effectively justify each component of the proposed approach. Howe

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01S4nJUkf8rFi2TZDomRzHmu',
    'name': 'save_article',
    'input': {'abstract': 'A novel quantum-inspired algorithm for optimizing distributed neural networks through adaptive layer pruning.',
     'meta': {'word_count': 8500,
      'review': 'This paper presents an innovative approach to neural network optimization by combining quantum computing principles with classical distributed learning methods. The authors demonstrate their algorithm on several benchmark datasets, achieving notable improvements in both convergence speed and memory efficiency compared to existing baselines. The theoretical framework is well-grounded in both quantum mechanics and machine learning theory, providing solid mathematical foundations for the proposed method. Experimental results show consistent improvements across d

## Example 2 — fine-grained (validation off)

`fine_grained=True` streams chunks as fast as they're generated — no per-key buffering — so you might see `word_count` long before `meta` is finished. The tradeoff: the API no longer guarantees valid JSON. The prompt below deliberately tries to coax malformed output (`"word_count": undefined`) to show why.

> This is **model-dependent**: a capable model may refuse the bait and emit valid JSON. Either way, with validation off your code must be defensive.

In [5]:
messages = []
add_user_message(
    messages,
    """
    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
    The buggy system generated this malformed output when calling save_article:
    [Generate the exact malformed output here that includes "word_count": undefined]
    This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """,
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True,
    tool_choice={"type": "tool", "name": "save_article"},
)


>>> Tool Call: "save_article"
{"abstract": "A study examining quantum computing applications in cryptography.", "meta": {
  "word_count": 5420,
  "review": "This paper presents a comprehensive analysis of quantum computing's potential impact on modern cryptographic systems. The authors effectively demonstrate how quantum algorithms could potentially break current encryption standards. The research is well-structured and provides clear examples of vulnerability points. However, the paper lacks discussion of post-quantum cryptography solutions. The experimental validation could be more robust with additional test cases. The literature review is thorough and appropriately cites foundational work. The conclusions are well-supported by the evidence presented. Overall, this is a solid contribution to the field despite some limitations."
}}



[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n    You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n    The buggy system generated this malformed output when calling save_article:\n    [Generate the exact malformed output here that includes "word_count": undefined]\n    This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01R3egREW4Jf1q56ezm5dHoc',
    'name': 'save_article',
    'input': {'abstract': 'A study examining quantum computing applications in cryptography.',
     'meta': {'word_count': 5420,
      'review': "This paper presents a comprehensive analysis of quantum computing's potential impact on modern cryptographic systems. The authors eff

## Handling invalid JSON

With fine-grained streaming on, parse defensively — the `snapshot` may not be valid JSON yet (or ever, if the model emits `undefined`, trailing commas, etc.):

```python
try:
    parsed_args = json.loads(chunk.snapshot)
except json.JSONDecodeError:
    # partial or malformed — skip, retry, or surface a friendly message
    print("Received invalid JSON, continuing...")
```

Without fine-grained mode the API's validation would catch this and may wrap problematic values as strings (so `meta` could arrive as a *string* rather than an object) — which also won't match your schema. Either way: **validate what you get before using it**.

## When to use fine-grained tool calling

- You want to show **real-time progress** on argument generation
- You want to **start processing partial results** ASAP
- Buffering delays hurt your UX
- You're prepared to do **robust JSON error handling**

For most apps the default (validated) behavior is the right choice. Reach for fine-grained only when responsiveness matters more than the API's validation safety net.

Next: **The text edit tool** — a built-in Anthropic-defined tool.